# US Revenue Forecast v2 — 단계별 테스트 노트북

**v6 변경 사항** (2026-04-30):
- forecast_index 패치 v6: 53주 fiscal year 보정
- 13W ticker 의 한 번의 98일 gap 도 정확 인식 (NVDA, AAPL 등 ~250개)

**v5 변경 사항** (2026-04-30):
- forecast_index 패치 v5: 13W fiscal calendar 정확 처리
- `infer_freq_alias` 와 `make_forecast_index` 가 `pd.infer_freq` 의 정교한 freq 를 활용
- 영향: 약 352개 ticker (NVDA, AAPL, JNJ, COST 등) 의 forecast 첫 분기 정확화

**v2 변경 사항** (2026-04-30):
- `DATA_SOURCE` 를 'fmp' 로 단일화 (이전 'db' 옵션 deprecated)
- `_fmp_fetch_income` 만 사용 — FMP API 직접 조회로 fiscal calendar 정보 정확 보존
- 13W fiscal calendar ticker (AAP, PVH 등 250개) 정확 처리
- DB(US_IS_from_FMP) 의존성 제거

**v1 → v2 마이그레이션**:
- `us_revenue_forecast_data` 테이블 정리 필요 (별도 cleanup 노트북 참조)
- DCFModel: v10.5 패치 유지 (in-memory 에서 fiscal end → calendar Q end 정규화)

**노트북 구성**:
- Cell 1: 환경 설정 & 경로 자동 감지
- Cell 2: 모듈 Import
- Cell 3: 파라미터 설정 (★ DATA_SOURCE='fmp' 고정)
- Cell 4: DB 연결 테스트
- Cell 5: 재무 데이터 추출 함수 정의 (FMP API 단일화)
- Cell 6: 단일 티커 데이터 추출 테스트
- Cell 7: 예측 함수 정의
- Cell 8: 단일 티커 예측 테스트
- Cell 9: Long-format 변환 함수 정의
- Cell 10: Long-format 변환 테스트
- Cell 11: DB 테이블 생성 & 저장 함수 정의
- Cell 12: 단일 티커 저장 테스트
- Cell 13: 배치 실행 (전체 / 특정 티커 / 구간 지정)
- Cell 14: 저장 결과 조회
- Cell 15: 오염 데이터 삭제 & 재예측 (옵션)

**의존성**:
- DATA/config.py — DB 연결 정보 (us_revenue_forecast_data 테이블만 사용)
- DATA/us_target_ticker_list_2000.py — 2000개 티커 리스트
- DATA/universal_ts_forecast_function_v2.py — 시계열 예측 모델

**주요 기능**:
- 5개 모델 (SARIMA, ETS, Prophet, LSTM, Theta) + 앙상블 (SARIMA+ETS+Theta)
- 8분기 예측 (2년)
- 음수 매출 자동 제외
- 최소 28분기 관측치 필요
- 메모리 최적화 (티커별 즉시 저장)
- DB 중복 자동 방지 ((ticker, item, date, model, forecast_date) UNIQUE)


## Cell 1 · 환경 설정 & 경로 자동 감지

노트북(Hoyoung_Park) / 데스크탑(82108) 어느 환경에서 실행해도  
`DATA` 폴더를 자동으로 찾아 `sys.path`에 추가합니다.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ───────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",          # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",   # 데스크탑
]

def _setup_path() -> str:
    """
    프로젝트 루트(DATA/ 폴더의 부모)를 탐색해 sys.path 에 추가합니다.
    탐색 순서:
      1) __file__ 또는 cwd 기준 상위 경로 중 DATA/ 를 포함하는 첫 번째 경로
      2) _CANDIDATE_ROOTS 에서 실존하는 첫 번째 경로
    """
    try:
        start = Path(__file__).resolve().parent
    except NameError:          # 노트북 환경 — __file__ 없음
        start = Path.cwd()

    # 현재 경로부터 상위로 올라가며 DATA/ 탐색
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    # cwd 탐색에서 못 찾으면 후보 경로 시도
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다.\n"
        "_CANDIDATE_ROOTS 목록을 현재 환경에 맞게 수정하거나 "
        "노트북을 프로젝트 루트 아래에서 실행하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트  : {_ROOT}")
print(f"[확인] DATA 경로     : {os.path.join(_ROOT, 'DATA')}")
print(f"[확인] sys.path[0]   : {sys.path[0]}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로     : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
[확인] sys.path[0]   : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\미국전기업_매출_예측


## Cell 2 · 모듈 Import

`DATA` 폴더 내 세 모듈과 외부 라이브러리를 불러옵니다.

In [2]:
# ── 내부 모듈 (DATA 폴더) ─────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima,
    forecast_ets,
    forecast_prophet,
    forecast_lstm,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq,
    clear_memory,
)

# ── 외부 라이브러리 ───────────────────────────────────────────
import gc
import traceback
from typing import Optional, List      # Python 3.9 호환 타입 힌트
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import text
from IPython.display import display

# ── 로그 유틸 (config 에 log 가 없는 경우 자체 정의) ──────────
try:
    from DATA.config import log
except ImportError:
    def log(tag: str, msg: str):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}][{tag}] {msg}")

print(f"[OK] 모든 모듈 Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST 길이: {len(DEFAULT_TICKER_LIST):,}개")


[OK] 모든 모듈 Import 완료
[OK] DEFAULT_TICKER_LIST 길이: 2,000개


## Cell 3 · 파라미터 설정

항목·기간·모델·배치 등 전역 파라미터를 여기서만 수정합니다.

In [3]:
# ════════════════════════════════════════════════════════════
#  ★ 파라미터 — 필요에 따라 이 셀만 수정하세요 ★
# ════════════════════════════════════════════════════════════

# ── FMP API ──────────────────────────────────────────────
FMP_API_KEY    = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE_URL   = "https://financialmodelingprep.com/api/v3"
FMP_MAX_RETRY  = 3
FMP_SLEEP_SEC  = 0.35

# ── 데이터 소스 ───────────────────────────────────────────
# v2: FMP API 직접 조회로 단일화
#  · 13W fiscal calendar ticker (AAP, PVH 등 250개) 도 정확하게 처리
#  · DB(US_IS_from_FMP) 의존성 제거
#  · Starter plan 250 calls/min — 2000 ticker × 1 call ≈ 8분
DATA_SOURCE    = "fmp"   # ← 'fmp' 고정 (이전 'db' 옵션은 deprecated)

DEST_TABLE = "us_revenue_forecast_data" # 예측 결과 저장 테이블

# ── 재무 항목 ─────────────────────────────────────────────
# 예: "sale" (매출) / "opi" (영업이익) / "ni" (순이익) / "ebitda" 등
ITEM       = "sale"

# ── 예측 설정 ─────────────────────────────────────────────
HORIZON    = 8    # 예측 분기 수 (default 8 = 2년)
MIN_OBS    = 28   # 최소 관측 분기 수 (28 = 7년 × 4분기)

# ── 모델 선택 ─────────────────────────────────────────────
ALL_MODELS      = ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]
ENSEMBLE_MODELS = ["SARIMA", "ETS", "Theta"]   # 앙상블 구성 모델

# ── 예측 실행일 ───────────────────────────────────────────
FORECAST_DATE = datetime.now().strftime("%Y-%m-%d")

# ════════════════════════════════════════════════════════════
print("[파라미터 확인]")
print(f"  DATA_SOURCE  = {DATA_SOURCE}  (★ FMP API 단일화)")
print(f"  DEST_TABLE   = {DEST_TABLE}")
print(f"  ITEM         = {ITEM}")
print(f"  HORIZON      = {HORIZON}분기")
print(f"  MIN_OBS      = {MIN_OBS}개")
print(f"  ALL_MODELS   = {ALL_MODELS}")
print(f"  ENSEMBLE     = {ENSEMBLE_MODELS}")
print(f"  FORECAST_DATE= {FORECAST_DATE}")


[파라미터 확인]
  DATA_SOURCE  = fmp  (★ FMP API 단일화)
  DEST_TABLE   = us_revenue_forecast_data
  ITEM         = sale
  HORIZON      = 8분기
  MIN_OBS      = 28개
  ALL_MODELS   = ['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']
  ENSEMBLE     = ['SARIMA', 'ETS', 'Theta']
  FORECAST_DATE= 2026-04-30


## Cell 4 · DB 연결 테스트

In [4]:
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("[OK] DB 연결 성공")
    print(f"     host={db_info.get('host')}  port={db_info.get('port')}  db={db_info.get('database')}")
except Exception as e:
    print(f"[FAIL] DB 연결 실패: {e}")


[OK] DB 연결 성공
     host=192.168.0.230  port=3307  db=investar


## Cell 5 · 재무 데이터 추출 함수 정의

`US_IS_from_FMP` → ticker + item 기준 분기 시계열 추출  

**처리 흐름**
1. ticker / item 기준으로 value, date, period, date_month 추출  
2. 날짜 파싱 및 오름차순 정렬  
3. 월별 중복 제거 (같은 월 → 마지막 행, 단독 행은 보존)  
4. 분기 단위 재집계 (같은 분기 → 마지막 행, 분기말 날짜로 통일)  
5. MIN_OBS 미달 시 ValueError

In [5]:
import time as _time
import requests as _requests


def _fmp_fetch_income(
    ticker: str,
    limit: int = 40,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    FMP API 에서 손익계산서를 직접 조회합니다.
    반환: columns [date, report_date, period, date_month, value]

    ★ 중요: FMP API 의 date 는 fiscal Q end 그대로 보존됩니다.
       - 12월 fiscal year ticker  → calendar Q end 와 동일
       - 비-12월 fiscal year ticker → fiscal Q end (예: 4/30)
       - 13W fiscal calendar ticker → fiscal Q end (예: 4/23 토요일)
       이 형식은 시계열 모델 (SARIMA 등) 의 정확한 분기 간격을 보장합니다.
    """
    url = f"{FMP_BASE_URL}/income-statement/{ticker}"
    params = {"period": period, "limit": limit, "apikey": FMP_API_KEY}

    for k in range(FMP_MAX_RETRY):
        try:
            r = _requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                _time.sleep(1.5 + k)
                continue
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and "Error Message" in data:
                raise ValueError(data["Error Message"])
            break
        except Exception as e:
            if k == FMP_MAX_RETRY - 1:
                raise RuntimeError(f"FMP 조회 실패 ({ticker}): {e}")
            _time.sleep(FMP_SLEEP_SEC + k * 0.5)

    if not data or not isinstance(data, list):
        return pd.DataFrame()

    df = pd.DataFrame(data)
    if df.empty or "revenue" not in df.columns:
        return pd.DataFrame()

    # FMP 컬럼 → 내부 표준 컬럼으로 변환
    df["date"]        = pd.to_datetime(df["date"],         errors="coerce")
    df["report_date"] = pd.to_datetime(df.get("fillingDate", df.get("acceptedDate", pd.NaT)), errors="coerce")
    df["period"]      = df.get("period", pd.NA)
    # date_month: date 컬럼의 월 첫날 (FMP date = 분기말이므로 그대로 사용)
    df["date_month"]  = df["date"].dt.to_period("M").dt.to_timestamp()
    df["value"]       = pd.to_numeric(df["revenue"], errors="coerce")

    df = (
        df[["date", "report_date", "period", "date_month", "value"]]
        .dropna(subset=["date", "value"])
        .sort_values("date")
        .reset_index(drop=True)
    )
    return df


def _clean_series(df: pd.DataFrame, ticker: str, item: str) -> pd.DataFrame:
    """
    추출된 원시 DataFrame 을 정제합니다.
    - date 기준 중복 제거 (같은 분기말이 여러 번 → 마지막 유지)
    - 음수 매출 검사
    - 최소 관측치 검사
    """
    df = df.copy()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date", "value"]).sort_values("date").reset_index(drop=True)

    # ── FMP date 는 fiscal Q end 그대로 사용 ──────────────
    # 단, 같은 date 가 중복으로 들어온 경우 마지막 행 유지
    df = (
        df.groupby("date", sort=True)
          .last()
          .reset_index()
    )

    return df


def fetch_financial_series(
    engine,
    ticker: str,
    item: str    = "sale",
    min_obs: int = 28,
) -> pd.DataFrame:
    """
    FMP API 에서 분기 매출 시계열을 추출합니다.
    
    v2 변경 사항:
      - DATA_SOURCE='db' 옵션 제거 — FMP API 단일화
      - DB(US_IS_from_FMP) 의 _qend 변환 로직 제거 (fiscal calendar 정보 손실 방지)
      - 13W fiscal calendar ticker 정확 처리
    
    반환: columns [date, report_date, period, date_month, value]
    """
    # FMP API 직접 조회
    df = _fmp_fetch_income(ticker, limit=max(min_obs + 10, 40))
    if df.empty:
        raise ValueError(f"[{ticker}] FMP에서 '{item}' 데이터 없음")
    _time.sleep(FMP_SLEEP_SEC)  # API 호출 간격

    # ── 공통 정제 ─────────────────────────────────────────────
    df = _clean_series(df, ticker, item)

    # ── 음수 매출 검사 ────────────────────────────────────────
    if (df["value"] < 0).any():
        neg_dates = df.loc[df["value"] < 0, "date"].dt.date.tolist()
        raise ValueError(
            f"[{ticker}] '{item}' 음수 매출 존재 → 예측 제외 "
            f"(음수 분기: {neg_dates})"
        )

    # ── 최소 관측치 검사 ──────────────────────────────────────
    if len(df) < min_obs:
        raise ValueError(
            f"[{ticker}] '{item}' 관측치 부족: {len(df)}개 < 최소 {min_obs}개"
        )

    return df


print("[OK] fetch_financial_series 재정의 완료 (v2)")
print(f"     DATA_SOURCE = '{DATA_SOURCE}' (FMP API 단일화)")
print(f"     → 13W fiscal calendar ticker 도 정확 처리")
print(f"     → fiscal calendar 정보 보존 (시계열 모델 정확성 ↑)")


[OK] fetch_financial_series 재정의 완료 (v2)
     DATA_SOURCE = 'fmp' (FMP API 단일화)
     → 13W fiscal calendar ticker 도 정확 처리
     → fiscal calendar 정보 보존 (시계열 모델 정확성 ↑)


In [6]:
# ── Cell 6-진단 : DB 원본 데이터 vs 추출 결과 비교 ──────────
# 최신 분기 누락 여부를 확인하는 진단 셀
from sqlalchemy import text as _text

DIAG_TICKER = "AAP"   # ← 확인할 티커

print(f"[진단] {DIAG_TICKER} — DB 원본 vs fetch 결과 비교")
print("=" * 60)

# DB 원본 (최근 5행)
with engine.connect() as conn:
    raw = pd.read_sql(
        _text("""
            SELECT date, report_date, period, date_month, value
            FROM   US_IS_from_FMP
            WHERE  ticker = :ticker AND item = :item
              AND  value IS NOT NULL
            ORDER  BY date DESC
            LIMIT  6
        """),
        conn,
        params={"ticker": DIAG_TICKER, "item": ITEM}
    )

print("[DB 원본] 최근 6행 (date 내림차순):")
display(raw)

# fetch_financial_series 결과 (최근 5행)
try:
    diag_df = fetch_financial_series(engine, DIAG_TICKER, item=ITEM, min_obs=MIN_OBS)
    print(f"\n[fetch 결과] 최근 5행 (총 {len(diag_df)}분기):")
    display(diag_df.tail(5))
    print(f"\n  → 마지막 분기 date : {diag_df['date'].iloc[-1].date()}")
    print(f"  → 기대값           : 2025-12-31 (NVDA 2025Q4)")
    ok = diag_df['date'].iloc[-1].date().isoformat() == '2026-03-31'
    print(f"  → {'✅ 정상' if ok else '❌ 여전히 누락 — 추가 확인 필요'}")
except Exception as e:
    print(f"[오류] {e}")


[진단] AAP — DB 원본 vs fetch 결과 비교
[DB 원본] 최근 6행 (date 내림차순):


,date,report_date,period,date_month,value
0,2026-01-31,2025-10-04,Q3,2025-10,2.003000e+09
1,2026-01-31,2026-01-03,Q4,2026-01,1.973000e+09
2,2025-12-31,2025-10-04,Q3,2025-10,2.003000e+09
3,2025-12-31,2025-10-04,Q3,2025-10,2.003000e+09
4,2025-11-30,2025-10-04,Q3,2025-10,2.003000e+09
5,2025-11-30,2025-10-04,Q3,2025-10,2.003000e+09



[fetch 결과] 최근 5행 (총 40분기):


,date,report_date,period,date_month,value
35,2024-12-28,2025-02-26,Q4,2024-12-01,1996025000
36,2025-04-19,2025-05-22,Q1,2025-04-01,2583000000
37,2025-07-12,2025-08-14,Q2,2025-07-01,2010000000
38,2025-10-04,2025-10-30,Q3,2025-10-01,2036000000
39,2026-01-03,2026-02-13,Q4,2026-01-01,1973000000



  → 마지막 분기 date : 2026-01-03
  → 기대값           : 2025-12-31 (NVDA 2025Q4)
  → ❌ 여전히 누락 — 추가 확인 필요


## Cell 6 · 단일 티커 데이터 추출 테스트

`TEST_TICKER` 를 원하는 티커로 변경해서 테스트하세요.

In [7]:
TEST_TICKER = DIAG_TICKER   # ← 테스트할 티커

try:
    src_df = fetch_financial_series(
        engine, TEST_TICKER, item=ITEM, min_obs=MIN_OBS
    )
    print(f"[OK] {TEST_TICKER} '{ITEM}' 추출 성공: {len(src_df)}분기")
    print(f"     기간: {src_df['date'].iloc[0].date()} ~ {src_df['date'].iloc[-1].date()}")
    display(src_df.tail(8))
except Exception as e:
    print(f"[FAIL] {e}")


[OK] AAP 'sale' 추출 성공: 40분기
     기간: 2016-04-23 ~ 2026-01-03


,date,report_date,period,date_month,value
32,2024-04-20,2024-05-30,Q1,2024-04-01,2772000000
33,2024-07-13,2024-08-22,Q2,2024-07-01,2178000000
34,2024-10-05,2024-11-14,Q3,2024-10-01,2148000000
35,2024-12-28,2025-02-26,Q4,2024-12-01,1996025000
36,2025-04-19,2025-05-22,Q1,2025-04-01,2583000000
37,2025-07-12,2025-08-14,Q2,2025-07-01,2010000000
38,2025-10-04,2025-10-30,Q3,2025-10-01,2036000000
39,2026-01-03,2026-02-13,Q4,2026-01-01,1973000000


## Cell 7 · 예측 함수 정의

- `make_forecast_index` : 마지막 실제값 다음 분기부터 HORIZON 개 날짜 생성  
- `forecast_one_ticker` : 5개 모델 순차 실행, 메모리 추적 포함

In [8]:
def make_forecast_index(
    last_date: pd.Timestamp,
    horizon: int,
    freq: str = "QE",
) -> pd.DatetimeIndex:
    """
    last_date 다음 분기부터 horizon 개의 날짜 인덱스를 생성합니다.
    freq : infer_freq_alias() 가 반환하는 값 (Q, QE, QS 등)
    """
    _freq = freq if freq else "QE"
    try:
        idx = pd.date_range(
            start   = last_date + pd.tseries.frequencies.to_offset(_freq),
            periods = horizon,
            freq    = _freq,
        )
    except Exception:
        # fallback: 3개월 간격으로 직접 생성
        idx = pd.date_range(
            start   = last_date + pd.DateOffset(months=3),
            periods = horizon,
            freq    = "QE",
        )
    return idx


def forecast_one_ticker(
    y: pd.Series,
    ticker: str,
    horizon: int,
    models: list,
) -> dict:
    """
    단일 티커의 시계열 y 에 대해 지정 모델들로 예측을 수행합니다.

    실제 함수 시그니처 (universal_ts_forecast_function_v2.py 기준):
      forecast_sarima  : (y, forecast_horizon, seasonal_period=int)
      forecast_ets     : (y, forecast_horizon, m=int)
      forecast_prophet : (y, forecast_horizon, m=int)
      forecast_lstm    : (y, forecast_horizon)           ← m 파라미터 없음
      forecast_theta   : (y, forecast_horizon, m=int)

    Parameters
    ----------
    y       : DatetimeIndex 를 가진 분기 시계열 (Series)
    ticker  : 종목 코드 (로그 출력용)
    horizon : 예측 분기 수
    models  : 사용할 모델 목록  ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]

    Returns
    -------
    dict  {model_name: {"forecast": array, "spec": dict} or {"error": str}}
    """
    import psutil, os
    proc = psutil.Process(os.getpid())

    def _mem_mb():
        return proc.memory_info().rss / 1024 / 1024

    freq    = infer_freq_alias(y.index)
    sp      = seasonal_periods_from_freq(freq)   # 분기=4, 월=12
    results = {}

    # ── 각 함수의 실제 파라미터명에 맞춰 호출 ─────────────────────
    def _call(model_name):
        if model_name == "SARIMA":
            # forecast_sarima(y, forecast_horizon, seasonal_period=)
            return forecast_sarima(y, horizon, seasonal_period=sp)
        elif model_name == "ETS":
            # forecast_ets(y, forecast_horizon, m=)
            return forecast_ets(y, horizon, m=sp)
        elif model_name == "Prophet":
            # forecast_prophet(y, forecast_horizon, m=)
            return forecast_prophet(y, horizon, m=sp)
        elif model_name == "LSTM":
            # forecast_lstm(y, forecast_horizon)  ← m 파라미터 없음
            return forecast_lstm(y, horizon)
        elif model_name == "Theta":
            # forecast_theta(y, forecast_horizon, m=)
            return forecast_theta(y, horizon, m=sp)
        else:
            raise ValueError(f"알 수 없는 모델: {model_name}")

    for model_name in models:
        log(ticker, f"  [{model_name}] 시작  (메모리: {_mem_mb():.1f} MB)")
        try:
            res = _call(model_name)
            results[model_name] = res
            fc_arr = np.asarray(res.get("forecast", []))
            if len(fc_arr) > 0:
                log(ticker, f"  [{model_name}] 완료  첫값={fc_arr[0]:.2e} (메모리: {_mem_mb():.1f} MB)")
            else:
                log(ticker, f"  [{model_name}] 오류응답: {res}")
        except Exception as e:
            log(ticker, f"  [{model_name}] 오류: {e}")
            results[model_name] = {"error": str(e)}
        finally:
            gc.collect()

    return results

print("[OK] make_forecast_index / forecast_one_ticker 함수 정의 완료")
print("  SARIMA  : forecast_sarima(y, forecast_horizon, seasonal_period=sp)")
print("  ETS     : forecast_ets(y, forecast_horizon, m=sp)")
print("  Prophet : forecast_prophet(y, forecast_horizon, m=sp)")
print("  LSTM    : forecast_lstm(y, forecast_horizon)")
print("  Theta   : forecast_theta(y, forecast_horizon, m=sp)")


[OK] make_forecast_index / forecast_one_ticker 함수 정의 완료
  SARIMA  : forecast_sarima(y, forecast_horizon, seasonal_period=sp)
  ETS     : forecast_ets(y, forecast_horizon, m=sp)
  Prophet : forecast_prophet(y, forecast_horizon, m=sp)
  LSTM    : forecast_lstm(y, forecast_horizon)
  Theta   : forecast_theta(y, forecast_horizon, m=sp)


## Cell 7.5 (UPDATED v8) · forecast_index 패치 v8 — 단순한 +3 month 방식 ★

### v7 → v8 변경 사유

호영님의 결정적 통찰:
> "분기별 날짜의 하드코딩이 그렇게 어렵나요? 3월말-6월말-9-12, 1-4-7-10, 2-5-8-11, ... 12 가지 경우만 처리하면 됩니다."

이 직관이 정확합니다. v5 ~ v7 의 복잡한 freq 추론 (`pd.infer_freq`, dow 분석, gap 분석) 모두 **불필요**.

### 핵심 통찰

**DCFModel 이 어차피 모든 dates 를 calendar Q end 로 정규화** 합니다. 그러므로 forecast 도 calendar Q end 로 통일하면 자연 일치.

### 단순 로직

```python
def make_forecast_index_v5(last_date, horizon=8):
    # last_date 의 month 에서 +3 씩 진행하며 그 month 의 마지막 날
    dates = []
    m, y = last_date.month, last_date.year
    for i in range(horizon):
        m += 3
        if m > 12:
            m -= 12; y += 1
        next_end = pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
        dates.append(next_end)
    return pd.DatetimeIndex(dates)
```

### 모든 fiscal pattern 자동 처리

| Fiscal year end | last_date | forecast 첫 |
|---|---|---|
| 12월 (정상) | 2026-03-31 | 2026-06-30 |
| 9월 (AAPL) | 2025-12-27 | 2026-03-31 |
| 1월 (NVDA) | 2026-01-25 | 2026-04-30 |
| 2월 (COST) | 2026-02-15 | 2026-05-31 |
| 8월 (AZO) | 2026-02-14 | 2026-05-31 |
| 4월 (WLY) | 2026-01-31 | 2026-04-30 |

모든 케이스에서:
- `forecast 첫 분기` 가 `actual 마지막` 과 다른 calendar Q
- CONFLICT 없음
- DCFModel 의 정규화와 자연 호환

### v8 의 장점

| 측면 | v7 | v8 |
|---|---|---|
| 코드 복잡도 | 100+ 줄 | 20 줄 |
| edge case 처리 | 4단계 분류 | 자동 |
| 53주 fiscal year | 별도 처리 | 영향 없음 |
| 84/112일 혼합 | 범위 확장 필요 | 영향 없음 |
| KO, WLY 같은 outlier | 작동 안 함 | **작동** |

### 사용법

이 셀이 Cell 7.5 의 v7 을 대체. 이전과 동일하게 Cell 7 다음에 실행.


In [17]:
# ════════════════════════════════════════════════════════════════════
#  forecast_index 패치 v8 — 단순한 +3 month 방식
# ════════════════════════════════════════════════════════════════════
#
#  v5/v6/v7 의 복잡한 freq 추론 모두 폐기.
#  
#  핵심 통찰:
#    DCFModel 이 어차피 모든 dates 를 calendar Q end 로 정규화.
#    forecast 도 calendar Q end 로 통일하면 자연 일치.
#
#  단순 로직:
#    next quarter end = last_date 의 month + 3 의 month-end
#    
#    12 가지 가능한 fiscal pattern 모두 자연스럽게 처리:
#    - 12월 fiscal (정상): 3-end → 6-end → 9-end → 12-end
#    - 1월 fiscal (NVDA):  1-end → 4-end → 7-end → 10-end → 1-end
#    - 2월 fiscal (COST):  2-end → 5-end → 8-end → 11-end → 2-end
#    - 8월 fiscal (AZO):   2-end → 5-end → 8-end → 11-end (last 2/14 → next 5/31)
#    - 9월 fiscal (AAPL): 12-end → 3-end → 6-end → 9-end
#    - 등등
#
#  검증:
#    - 모든 fiscal pattern 에서 forecast 첫 분기가 다음 calendar Q
#    - actual 마지막 분기와 같은 calendar Q 충돌 없음
#    - 53주 fiscal year, 84/112일 혼합 등 edge case 모두 자동 처리
#
#  사용법:
#    Cell 7.5 의 v7 코드를 v8 로 교체.
#    훨씬 단순하고 정확함.
# ════════════════════════════════════════════════════════════════════

import pandas as pd


def infer_freq_alias_v5(index):
    """
    이 함수는 호환성을 위해 유지하지만, make_forecast_index_v5 는 사용하지 않음.
    단순히 'QE' 반환 (의미 없는 placeholder).
    """
    return "QE"


def make_forecast_index_v5(last_date, horizon, freq=None):
    """
    last_date 다음 분기부터 horizon 개의 calendar month-end dates 생성.
    
    핵심: last_date 의 month 에서 +3 씩 진행하며 그 month 의 마지막 날 사용.
    
    예시:
      NVDA: last=2026-01-25 (Sun, fiscal Q4 FY26)
        → 2026-04-30, 2026-07-31, 2026-10-31, 2027-01-31, ...
      AAPL: last=2025-12-27 (Sat, fiscal Q1 FY26)
        → 2026-03-31, 2026-06-30, 2026-09-30, 2025-12-31, ...
      Calendar: last=2026-03-31
        → 2026-06-30, 2026-09-30, 2026-12-31, 2027-03-31, ...
      AZO:  last=2026-02-14 (Sat, fiscal Q2 FY26)
        → 2026-05-31, 2026-08-31, 2026-11-30, 2027-02-28, ...
    
    Parameters:
    -----------
    last_date : pd.Timestamp
        actual 시계열의 마지막 분기 end (어떤 fiscal calendar 든 무관)
    horizon : int
        예측 분기 수 (기본 8)
    freq : str
        호환성 위해 받지만 사용 안 함
    
    Returns:
    --------
    pd.DatetimeIndex
        last_date 의 month + 3 부터 시작하는 calendar month-end dates
    """
    dates = []
    m = last_date.month
    y = last_date.year
    
    for i in range(horizon):
        m += 3
        if m > 12:
            m -= 12
            y += 1
        # 그 month 의 마지막 날 (calendar)
        next_end = pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
        dates.append(next_end)
    
    return pd.DatetimeIndex(dates)


# ★ 모듈 + 노트북 namespace 둘 다 patch
import DATA.universal_ts_forecast_function_v2 as _ufm
_ufm.infer_freq_alias = infer_freq_alias_v5
_ufm.make_forecast_index = make_forecast_index_v5

infer_freq_alias = infer_freq_alias_v5
make_forecast_index = make_forecast_index_v5


# ════════════════════════════════════════════════════════════════════
#  자가 검증
# ════════════════════════════════════════════════════════════════════
print("[OK] forecast_index 패치 v8 적용 — 단순한 +3 month 방식")
print()
print("  핵심: DCFModel 이 어차피 calendar Q 로 정규화하니 forecast 도 통일")
print("  → fiscal calendar 다양성 (12W/13W/16W, dow 등) 모두 자동 처리")
print()
print("[자가 검증 — 다양한 fiscal pattern]")
print()

test_cases = [
    ("12월 fiscal (대부분)",  "2026-03-31"),
    ("9월 fiscal (AAPL)",     "2025-12-27"),
    ("1월 fiscal (NVDA)",     "2026-01-25"),
    ("2월 fiscal (COST)",     "2026-02-15"),
    ("8월 fiscal (AZO)",      "2026-02-14"),
    ("4월 fiscal (WLY)",      "2026-01-31"),
    ("5월 fiscal (CSCO)",     "2026-01-25"),
    ("12월 fiscal (JNJ)",     "2026-03-29"),
]

all_ok = True
for label, last_str in test_cases:
    last = pd.Timestamp(last_str)
    fc = make_forecast_index_v5(last, 8)
    
    same_q = fc[0].to_period("Q") == last.to_period("Q")
    if same_q:
        all_ok = False
    status = "✗" if same_q else "✓"
    
    print(f"  {status} {label:28s}  last={last.date()}  →  fc 첫={fc[0].date()}")

print()
if all_ok:
    print("  ✓ 모든 fiscal pattern 에서 충돌 없음")
else:
    print("  ✗ 일부 패턴 충돌 — 추가 검토 필요")


[OK] forecast_index 패치 v8 적용 — 단순한 +3 month 방식

  핵심: DCFModel 이 어차피 calendar Q 로 정규화하니 forecast 도 통일
  → fiscal calendar 다양성 (12W/13W/16W, dow 등) 모두 자동 처리

[자가 검증 — 다양한 fiscal pattern]

  ✓ 12월 fiscal (대부분)              last=2026-03-31  →  fc 첫=2026-06-30
  ✓ 9월 fiscal (AAPL)              last=2025-12-27  →  fc 첫=2026-03-31
  ✓ 1월 fiscal (NVDA)              last=2026-01-25  →  fc 첫=2026-04-30
  ✓ 2월 fiscal (COST)              last=2026-02-15  →  fc 첫=2026-05-31
  ✓ 8월 fiscal (AZO)               last=2026-02-14  →  fc 첫=2026-05-31
  ✓ 4월 fiscal (WLY)               last=2026-01-31  →  fc 첫=2026-04-30
  ✓ 5월 fiscal (CSCO)              last=2026-01-25  →  fc 첫=2026-04-30
  ✓ 12월 fiscal (JNJ)              last=2026-03-29  →  fc 첫=2026-06-30

  ✓ 모든 fiscal pattern 에서 충돌 없음


## Cell 8 · 단일 티커 예측 테스트

Cell 6 에서 추출한 `src_df` 를 사용합니다.  
모델별 예측값과 SARIMA 파라미터를 확인하세요.

In [10]:
# Cell 6 에서 src_df 가 정상 추출된 경우에만 실행
y = src_df.set_index("date")["value"].copy()
y.index = pd.DatetimeIndex(y.index)
y.name  = ITEM

print(f"예측 입력 시계열: {len(y)}분기  ({y.index[0].date()} ~ {y.index[-1].date()})")

# 테스트용 모델 — 빠른 확인이 필요하면 ['SARIMA', 'ETS', 'Theta'] 로 축소 가능
TEST_MODELS = ALL_MODELS

forecast_results = forecast_one_ticker(y, TEST_TICKER, HORIZON, TEST_MODELS)
freq             = infer_freq_alias(y.index)
forecast_index   = make_forecast_index(y.index[-1], HORIZON, freq)

print("\n[예측 결과 요약]")
for model_name, res in forecast_results.items():
    if "error" in res:
        print(f"  {model_name:<10}: 오류 → {res['error']}")
    else:
        fc  = np.asarray(res["forecast"])
        msg = f"  {model_name:<10}: {fc.round(0).tolist()}"
        if model_name == "SARIMA" and "spec" in res:
            spec = res["spec"]
            aic  = spec.get("ic_value", "")
            aic_str = f"  AIC={aic:.2f}" if isinstance(aic, float) else ""
            msg += f"  | order={spec.get('order')} seasonal={spec.get('seasonal_order')}{aic_str}"
        print(msg)

예측 입력 시계열: 40분기  (2016-04-23 ~ 2026-01-03)
[AAP]   [SARIMA] 시작  (메모리: 434.8 MB)
[메모리] forecast_sarima 실행 전: 434.75 MB
[메모리] find_best_sarima_params 실행 전: 434.77 MB
[메모리] find_best_sarima_params 실행 후: 446.79 MB (변화: +12.02 MB)
[메모리] forecast_sarima 실행 후: 446.89 MB (변화: +12.14 MB)
[AAP]   [SARIMA] 완료  첫값=2.42e+09 (메모리: 446.9 MB)
[AAP]   [ETS] 시작  (메모리: 447.0 MB)
[메모리] forecast_ets 실행 전: 446.96 MB
[메모리] forecast_ets 실행 후: 447.17 MB (변화: +0.21 MB)
[AAP]   [ETS] 완료  첫값=2.62e+09 (메모리: 447.2 MB)
[AAP]   [Prophet] 시작  (메모리: 447.2 MB)


22:23:28 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 447.17 MB


22:23:28 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 448.61 MB (변화: +1.44 MB)
[AAP]   [Prophet] 완료  첫값=-6.14e+08 (메모리: 448.6 MB)
[AAP]   [LSTM] 시작  (메모리: 448.6 MB)
[메모리] forecast_lstm 실행 전: 448.61 MB
[메모리] forecast_lstm 실행 후: 1625.94 MB (변화: +1177.33 MB)
[경고] 메모리 사용량이 크게 증가했습니다. 메모리 정리를 권장합니다.
[AAP]   [LSTM] 완료  첫값=2.40e+09 (메모리: 1625.9 MB)
[AAP]   [Theta] 시작  (메모리: 1625.9 MB)
[메모리] forecast_theta 실행 전: 1625.94 MB
[메모리] forecast_theta 실행 후: 1626.09 MB (변화: +0.16 MB)
[AAP]   [Theta] 완료  첫값=2.11e+09 (메모리: 1626.1 MB)

[예측 결과 요약]
  SARIMA    : [2420186882.0, 2072052158.0, 1946922745.0, 1883772846.0, 2281389013.0, 1899351496.0, 1846028768.0, 1711020953.0]  | order=(1, 0, 0) seasonal=(1, 0, 1, 12)  AIC=-55.56
  ETS       : [2616997790.0, 2170700186.0, 2022551556.0, 1998752722.0, 2612892960.0, 2082333253.0, 2049979369.0, 1892047363.0]
  Prophet   : [-614184507.0, -756134249.0, -2216533075.0, 2290360778.0, -2937000572.0, 2078879731.0, 1787331389.0, -2666093101.0]
  LSTM      : [2398195200.0, 2361723392.0, 2349718784.

## Cell 9 · Long-format 변환 함수 정의

actual + forecast(각 모델) + Ensemble → 하나의 long-format DataFrame

**저장 컬럼**

| 컬럼 | 설명 |
|------|------|
| ticker | 종목 코드 |
| item | 재무 항목 |
| date | 기준일(분기말) |
| period | 회계 분기(Q1~Q4/FY) — actual 행만 |
| date_month | 해당 분기 기간 — actual 행만 |
| data_type | `actual` / `forecast` |
| model | actual / SARIMA / ETS / Prophet / LSTM / Theta / Ensemble |
| value | 수치값 |
| forecast_date | 예측 실행일 |
| sarima_order | SARIMA (p,d,q) — SARIMA 행만 |
| sarima_seasonal_order | SARIMA (P,D,Q,m) — SARIMA 행만 |
| sarima_ic_value | SARIMA 최적 AIC — SARIMA 행만 |
| created_at | 레코드 생성 시각 |

In [11]:
def build_long_df(
    ticker: str,
    item: str,
    src_df: pd.DataFrame,
    forecast_results: dict,
    forecast_index: pd.DatetimeIndex,
    forecast_date: str,
    ensemble_models: list = None,
) -> pd.DataFrame:
    """
    실제값(actual) + 예측값(각 모델) + 앙상블 → long-format DataFrame.

    Parameters
    ----------
    ticker           : 종목 코드
    item             : 재무 항목
    src_df           : fetch_financial_series() 반환 DataFrame
    forecast_results : forecast_one_ticker() 반환 dict
    forecast_index   : 예측 날짜 DatetimeIndex
    forecast_date    : 예측 실행일 (str)
    ensemble_models  : 앙상블 구성 모델 목록 (None → ENSEMBLE_MODELS 전역 변수 사용)
    """
    if ensemble_models is None:
        ensemble_models = ENSEMBLE_MODELS

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows    = []

    # ── actual 행 ────────────────────────────────────────────
    for _, row in src_df.iterrows():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : row["date"].strftime("%Y-%m-%d"),
            "period"                : row.get("period"),
            "date_month"            : (
                row["date_month"].strftime("%Y-%m-%d")
                if pd.notna(row.get("date_month")) else None
            ),
            "data_type"             : "actual",
            "model"                 : "actual",
            "value"                 : round(float(row["value"]), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    # ── forecast 행 ──────────────────────────────────────────
    ensemble_bucket = {}   # { date_str : [val, ...] }

    for model_name, res in forecast_results.items():
        if "error" in res or "forecast" not in res:
            continue

        fc_arr = np.asarray(res["forecast"])
        spec   = res.get("spec", {})

        sarima_order    = None
        sarima_seasonal = None
        sarima_ic       = None
        if model_name == "SARIMA":
            sarima_order    = str(spec.get("order",          ""))
            sarima_seasonal = str(spec.get("seasonal_order", ""))
            raw_ic = spec.get("ic_value")
            if raw_ic is not None:
                try:
                    v = float(raw_ic)
                    sarima_ic = round(v, 4) if np.isfinite(v) else None
                except (TypeError, ValueError):
                    pass

        for i, dt in enumerate(forecast_index):
            if i >= len(fc_arr):
                break
            val    = float(fc_arr[i])
            dt_str = dt.strftime("%Y-%m-%d")

            rows.append({
                "ticker"                : ticker,
                "item"                  : item,
                "date"                  : dt_str,
                "period"                : None,
                "date_month"            : None,
                "data_type"             : "forecast",
                "model"                 : model_name,
                "value"                 : round(val, 6),
                "forecast_date"         : forecast_date,
                "sarima_order"          : sarima_order,
                "sarima_seasonal_order" : sarima_seasonal,
                "sarima_ic_value"       : sarima_ic,
                "created_at"            : now_str,
            })

            if model_name in ensemble_models:
                ensemble_bucket.setdefault(dt_str, []).append(val)

    # ── Ensemble 행 (SARIMA + ETS + Theta 평균) ─────────────
    for dt_str, vals in ensemble_bucket.items():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : dt_str,
            "period"                : None,
            "date_month"            : None,
            "data_type"             : "forecast",
            "model"                 : "Ensemble",
            "value"                 : round(float(np.mean(vals)), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    return pd.DataFrame(rows)

print("[OK] build_long_df 함수 정의 완료")


[OK] build_long_df 함수 정의 완료


## Cell 10 · Long-format 변환 테스트

In [12]:
long_df = build_long_df(
    ticker           = TEST_TICKER,
    item             = ITEM,
    src_df           = src_df,
    forecast_results = forecast_results,
    forecast_index   = forecast_index,
    forecast_date    = FORECAST_DATE,
)

print(f"[OK] long_df 생성: {len(long_df)}행")
print("\n모델별 행 수:")
display(long_df.groupby(["data_type", "model"]).size().reset_index(name="rows"))
print("\n샘플 (forecast 상위 5행):")
display(long_df[long_df["data_type"] == "forecast"].head())


[OK] long_df 생성: 88행

모델별 행 수:


,data_type,model,rows
0,actual,actual,40
1,forecast,ETS,8
2,forecast,Ensemble,8
3,forecast,LSTM,8
4,forecast,Prophet,8
5,forecast,SARIMA,8
6,forecast,Theta,8



샘플 (forecast 상위 5행):


,ticker,item,date,period,date_month,data_type,model,value,forecast_date,sarima_order,sarima_seasonal_order,sarima_ic_value,created_at
40,AAP,sale,2026-04-30,None,None,forecast,SARIMA,2.420187e+09,2026-04-30,"(1, 0, 0)","(1, 0, 1, 12)",-55.5565,2026-04-30 22:23:37
41,AAP,sale,2026-07-31,None,None,forecast,SARIMA,2.072052e+09,2026-04-30,"(1, 0, 0)","(1, 0, 1, 12)",-55.5565,2026-04-30 22:23:37
42,AAP,sale,2026-10-31,None,None,forecast,SARIMA,1.946923e+09,2026-04-30,"(1, 0, 0)","(1, 0, 1, 12)",-55.5565,2026-04-30 22:23:37
43,AAP,sale,2027-01-31,None,None,forecast,SARIMA,1.883773e+09,2026-04-30,"(1, 0, 0)","(1, 0, 1, 12)",-55.5565,2026-04-30 22:23:37
44,AAP,sale,2027-04-30,None,None,forecast,SARIMA,2.281389e+09,2026-04-30,"(1, 0, 0)","(1, 0, 1, 12)",-55.5565,2026-04-30 22:23:37


In [13]:
# long_df.to_csv(r"C:\reports\revenue_forecast_data.csv")

## Cell 11 · DB 테이블 생성 & 저장 함수 정의

**중복 판정 기준** : `(ticker, item, date, model, forecast_date)`  
→ 이미 존재하는 행은 건드리지 않고, 신규 행만 INSERT

In [14]:
# ── 테이블 CREATE (최초 1회) ──────────────────────────────────
CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{DEST_TABLE}` (
    id                    BIGINT       NOT NULL AUTO_INCREMENT,
    ticker                VARCHAR(20)  NOT NULL,
    item                  VARCHAR(30)  NOT NULL,
    date                  DATE         NOT NULL,
    period                VARCHAR(10)  DEFAULT NULL,
    date_month            DATE         DEFAULT NULL,
    data_type             VARCHAR(10)  NOT NULL COMMENT 'actual / forecast',
    model                 VARCHAR(20)  NOT NULL,
    value                 DOUBLE       DEFAULT NULL,
    forecast_date         DATE         NOT NULL,
    sarima_order          VARCHAR(30)  DEFAULT NULL,
    sarima_seasonal_order VARCHAR(30)  DEFAULT NULL,
    sarima_ic_value       DOUBLE       DEFAULT NULL,
    created_at            DATETIME     DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    UNIQUE KEY uq_main (ticker, item, date, model, forecast_date),
    INDEX idx_ticker      (ticker),
    INDEX idx_forecast_dt (forecast_date),
    INDEX idx_model       (model)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

def ensure_table(engine):
    """저장 테이블이 없으면 생성합니다."""
    with engine.begin() as conn:
        conn.execute(text(CREATE_TABLE_SQL))
    print(f"[OK] 테이블 '{DEST_TABLE}' 준비 완료")


def save_to_db(engine, long_df: pd.DataFrame, dest_table: str = DEST_TABLE) -> int:
    """
    long_df 를 DB 에 저장합니다.
    - 중복 기준: (ticker, item, date, model, forecast_date)
    - 기존 행 유지 + 신규 행만 INSERT

    Returns
    -------
    int  : 실제 삽입된 신규 행 수
    """
    if long_df is None or long_df.empty:
        return 0

    # ── 1. 기존 키 조회 ────────────────────────────────────
    ticker      = long_df["ticker"].iloc[0]
    item        = long_df["item"].iloc[0]
    fc_date_val = long_df["forecast_date"].iloc[0]

    check_sql = text(f"""
        SELECT CONCAT(ticker,'|',item,'|',date,'|',model,'|',forecast_date) AS uq_key
        FROM   `{dest_table}`
        WHERE  ticker        = :ticker
          AND  item          = :item
          AND  forecast_date = :fc_date
    """)

    with engine.connect() as conn:
        existing = pd.read_sql(
            check_sql, conn,
            params={"ticker": ticker, "item": item, "fc_date": fc_date_val}
        )
    existing_keys = set(existing["uq_key"].tolist()) if not existing.empty else set()

    # ── 2. 신규 행 필터링 ──────────────────────────────────
    new_df = long_df[
        ~long_df.apply(
            lambda r: f"{r['ticker']}|{r['item']}|{r['date']}|{r['model']}|{r['forecast_date']}"
            in existing_keys,
            axis=1,
        )
    ].copy()

    if new_df.empty:
        log(ticker, f"  [DB] 신규 행 없음 — 스킵")
        return 0

    # ── 3. INSERT ─────────────────────────────────────────
    new_df.to_sql(
        name       = dest_table,
        con        = engine,
        if_exists  = "append",
        index      = False,
        chunksize  = 500,
        method     = "multi",
    )
    log(ticker, f"  [DB] {len(new_df)}행 저장 완료")
    return len(new_df)

print("[OK] ensure_table / save_to_db 함수 정의 완료")


[OK] ensure_table / save_to_db 함수 정의 완료


## Cell 12 · 단일 티커 저장 테스트

In [15]:
# 테이블 생성 (최초 1회)
ensure_table(engine)

# Cell 10 의 long_df 저장
inserted = save_to_db(engine, long_df)
print(f"[OK] {TEST_TICKER} 저장 완료 — 삽입: {inserted}행")

# 저장 확인
with engine.connect() as conn:
    chk = pd.read_sql(
        text(f"""
            SELECT model, data_type, COUNT(*) AS cnt
            FROM   `{DEST_TABLE}`
            WHERE  ticker = :tk AND forecast_date = :fd
            GROUP  BY model, data_type
            ORDER  BY model
        """),
        conn,
        params={"tk": TEST_TICKER, "fd": FORECAST_DATE},
    )
display(chk)


[OK] 테이블 'us_revenue_forecast_data' 준비 완료
[AAP]   [DB] 48행 저장 완료
[OK] AAP 저장 완료 — 삽입: 48행


,model,data_type,cnt
0,actual,actual,40
1,Ensemble,forecast,25
2,ETS,forecast,25
3,LSTM,forecast,25
4,Prophet,forecast,25
5,SARIMA,forecast,25
6,Theta,forecast,25


## Cell 13 · 배치 실행 (전체 / 특정 티커 / 구간 지정)

### 실행 모드 선택

| 변수 | 설명 |
|------|------|
| `RUN_TICKERS` | `None` → DEFAULT_TICKER_LIST 전체 / `["AAPL", ...]` → 특정 티커만 |
| `TICKER_START` | 리스트 슬라이싱 시작 인덱스 (0부터, `None` = 처음) |
| `TICKER_END`   | 리스트 슬라이싱 끝 인덱스 (None = 끝까지) |
| `RUN_MODELS`   | 사용할 모델 목록 |

### 구간 예시
```python
TICKER_START, TICKER_END = 0,   500   # 1~500번째 티커
TICKER_START, TICKER_END = 500, 1000  # 501~1000번째 티커
TICKER_START, TICKER_END = None, None # 전체
```

### 메모리 전략
> 티커 1개 예측 → 즉시 DB 저장 → `clear_memory()` 호출  
> 이 방식이 배치(20개 누적) 방식보다 피크 메모리가 낮고 중단 시 손실도 최소화됩니다.

In [16]:
# """
# ====================================================================
#   DB Cleanup — forecast 데이터 모두 삭제 + actual 중복 제거
# ====================================================================
#
# 호영님 제안대로 "기존 DB 싹 갈아엎고 깨끗하게 다시 시작" 하는 스크립트.
#
# 작업 내용:
#   1. us_revenue_forecast_data 의 모든 forecast row 삭제
#   2. us_revenue_forecast_data 의 actual 중복 row 정리
#      (같은 date 에 여러 forecast_date 로 저장된 것 → 가장 최신만 keep)
#   3. 통계 출력
#
# 사용법:
#   forecast 노트북 v8 의 새 cell 에 붙여넣기 + 실행
#
# 주의:
#   · 백업이 자동 생성되지 않습니다.
#   · DELETE 실행 전 확인 prompt 있음.
#   · DCFModel 의 valuation 은 FMP API 직접 호출하므로 영향 없음.
# """
#
# # ════════════════════════════════════════════════════════════════════
# #  Step 0: 현재 상태 확인
# # ════════════════════════════════════════════════════════════════════
# from sqlalchemy import text
#
# print("=" * 70)
# print("  현재 us_revenue_forecast_data 상태")
# print("=" * 70)
#
# with engine.connect() as conn:
#     # 전체 row 수
#     total = conn.execute(text("SELECT COUNT(*) FROM us_revenue_forecast_data")).scalar()
#
#     # data_type 별
#     rows_by_type = conn.execute(text("""
#         SELECT data_type, COUNT(*) AS n
#         FROM us_revenue_forecast_data
#         GROUP BY data_type
#     """)).fetchall()
#
#     # 고유 ticker 수
#     n_tickers = conn.execute(text("""
#         SELECT COUNT(DISTINCT ticker) FROM us_revenue_forecast_data
#     """)).scalar()
#
# print(f"\n  총 row: {total:,}")
# print(f"  고유 ticker: {n_tickers:,}")
# print(f"\n  data_type 별:")
# for r in rows_by_type:
#     print(f"    {r[0]:<12s}: {r[1]:>10,}")
#
# # actual 중복 확인 (같은 ticker, date 에 여러 row)
# with engine.connect() as conn:
#     actual_dup = conn.execute(text("""
#         SELECT COUNT(*) FROM (
#             SELECT ticker, date, COUNT(*) AS n
#             FROM us_revenue_forecast_data
#             WHERE data_type = 'actual'
#             GROUP BY ticker, date
#             HAVING n > 1
#         ) AS dup
#     """)).scalar()
#
# print(f"\n  actual 중복 (ticker, date 기준): {actual_dup:,}")
#
# # ════════════════════════════════════════════════════════════════════
# #  Step 1: 사용자 확인
# # ════════════════════════════════════════════════════════════════════
# print()
# print("=" * 70)
# print("  실행할 작업")
# print("=" * 70)
# print()
# print("  1. 모든 forecast row 삭제")
# print("  2. actual 중복 정리 (같은 (ticker, date) 의 여러 row → 가장 최신 forecast_date 만 keep)")
# print()
# print("  ※ DCFModel 의 valuation 은 FMP API 직접 호출 — 영향 없음")
# print("  ※ forecast 노트북에서 새로 forecast 만들면 됨")
# print()
#
# ans = input("  실행하시겠습니까? (y/N): ").strip().lower()
# if ans != 'y':
#     print("  → 취소됨")
# else:
#     # ════════════════════════════════════════════════════════════════
#     #  Step 2: forecast 데이터 모두 삭제
#     # ════════════════════════════════════════════════════════════════
#     print()
#     print("[Step 2] forecast 데이터 삭제 중...")
#
#     with engine.begin() as conn:
#         deleted_fc = conn.execute(text("""
#             DELETE FROM us_revenue_forecast_data
#             WHERE data_type = 'forecast'
#         """))
#
#     print(f"  ✓ {deleted_fc.rowcount:,} forecast rows 삭제")
#
#     # ════════════════════════════════════════════════════════════════
#     #  Step 3: actual 중복 제거
#     # ════════════════════════════════════════════════════════════════
#     print()
#     print("[Step 3] actual 중복 정리 중...")
#
#     # 같은 (ticker, date, data_type='actual') 중 forecast_date 가 가장 큰 것만 keep
#     with engine.begin() as conn:
#         deleted_dup = conn.execute(text("""
#             DELETE t1 FROM us_revenue_forecast_data t1
#             INNER JOIN us_revenue_forecast_data t2
#             WHERE t1.ticker = t2.ticker
#               AND t1.date = t2.date
#               AND t1.data_type = 'actual'
#               AND t2.data_type = 'actual'
#               AND t1.forecast_date < t2.forecast_date
#         """))
#
#     print(f"  ✓ {deleted_dup.rowcount:,} actual 중복 rows 삭제")
#
#     # ════════════════════════════════════════════════════════════════
#     #  Step 4: 결과 확인
#     # ════════════════════════════════════════════════════════════════
#     print()
#     print("=" * 70)
#     print("  Cleanup 후 상태")
#     print("=" * 70)
#
#     with engine.connect() as conn:
#         new_total = conn.execute(text("SELECT COUNT(*) FROM us_revenue_forecast_data")).scalar()
#         rows_by_type_new = conn.execute(text("""
#             SELECT data_type, COUNT(*) AS n
#             FROM us_revenue_forecast_data
#             GROUP BY data_type
#         """)).fetchall()
#         actual_dup_new = conn.execute(text("""
#             SELECT COUNT(*) FROM (
#                 SELECT ticker, date, COUNT(*) AS n
#                 FROM us_revenue_forecast_data
#                 WHERE data_type = 'actual'
#                 GROUP BY ticker, date
#                 HAVING n > 1
#             ) AS dup
#         """)).scalar()
#
#     print(f"\n  총 row: {new_total:,}  (이전: {total:,}, 감소: {total - new_total:,})")
#     print(f"\n  data_type 별:")
#     for r in rows_by_type_new:
#         print(f"    {r[0]:<12s}: {r[1]:>10,}")
#     print(f"\n  actual 중복: {actual_dup_new:,}")
#
#     print()
#     print("=" * 70)
#     print("  ✓ Cleanup 완료")
#     print("=" * 70)
#     print()
#     print("  다음 단계:")
#     print("  1. forecast 노트북 v8 의 Cell 7.5 (v8 patch) 실행")
#     print("  2. Cell 13 에서 RUN_TICKERS = None, TICKER_START = 0, TICKER_END = None")
#     print("  3. 전체 2,000 ticker re-forecast (40-60분)")
#     print("  4. 검증")

  현재 us_revenue_forecast_data 상태

  총 row: 341,395
  고유 ticker: 1,944

  data_type 별:
    actual      :    153,170
    forecast    :    188,225

  actual 중복 (ticker, date 기준): 55,148

  실행할 작업

  1. 모든 forecast row 삭제
  2. actual 중복 정리 (같은 (ticker, date) 의 여러 row → 가장 최신 forecast_date 만 keep)

  ※ DCFModel 의 valuation 은 FMP API 직접 호출 — 영향 없음
  ※ forecast 노트북에서 새로 forecast 만들면 됨


[Step 2] forecast 데이터 삭제 중...
  ✓ 188,225 forecast rows 삭제

[Step 3] actual 중복 정리 중...
  ✓ 69,891 actual 중복 rows 삭제

  Cleanup 후 상태

  총 row: 83,279  (이전: 341,395, 감소: 258,116)

  data_type 별:
    actual      :     83,279

  actual 중복: 0

  ✓ Cleanup 완료

  다음 단계:
  1. forecast 노트북 v8 의 Cell 7.5 (v8 patch) 실행
  2. Cell 13 에서 RUN_TICKERS = None, TICKER_START = 0, TICKER_END = None
  3. 전체 2,000 ticker re-forecast (40-60분)
  4. 검증


In [21]:
# ════════════════════════════════════════════════════════════
#  배치 설정 — 여기를 수정하세요
# ════════════════════════════════════════════════════════════

# ── 특정 티커 지정 (None 이면 아래 구간/전체 사용) ────────────
# Optional[list] = Python 3.9 호환 (3.10+ 의 list | None 대신 사용)
# RUN_TICKERS = None          # type: Optional[list]
# RUN_TICKERS = ["STRL", "SIMO", "APH", "ISSC","FIX", "DY"]   # 특정 티커만

# ── 전체 리스트 구간 지정 (RUN_TICKERS=None 일 때 적용) ──────
TICKER_START = 100           # type: Optional[int]  # 시작 인덱스 (0부터)
TICKER_END   = 500          # type: Optional[int]  # 끝 인덱스 (exclusive, None=끝까지)
# 예: 0~499   → START=0,   END=500
# 예: 500~999 → START=500, END=1000
# 예: 전체    → START=None, END=None

# ── 모델 / 항목 / 예측 기간 ───────────────────────────────────
RUN_MODELS  = ["SARIMA", "ETS", "Theta"] #ALL_MODELS   # 또는 ["SARIMA", "ETS", "Theta"]  (빠른 실행)
RUN_ITEM    = ITEM
RUN_HORIZON = HORIZON
RUN_MIN_OBS = MIN_OBS

# ════════════════════════════════════════════════════════════
#  실행 대상 티커 목록 결정
# ════════════════════════════════════════════════════════════
if RUN_TICKERS is not None:
    tickers = RUN_TICKERS
    print(f"[모드] 특정 티커 지정: {tickers}")
else:
    tickers = DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]
    _s = TICKER_START if TICKER_START is not None else 0
    _e = TICKER_END   if TICKER_END   is not None else len(DEFAULT_TICKER_LIST)
    print(f"[모드] 구간 실행: index {_s} ~ {_e-1}  ({len(tickers)}개)")

total = len(tickers)

# ════════════════════════════════════════════════════════════
#  배치 실행
# ════════════════════════════════════════════════════════════
ensure_table(engine)

success, skipped, errored, neg_skipped = 0, 0, 0, 0
skip_list, error_list, neg_skip_list   = [], [], []

log("BATCH", "=" * 70)
log("BATCH", f"시작  | 티커 {total}개 | 항목: {RUN_ITEM} | 예측기간: {RUN_HORIZON}분기")
log("BATCH", f"모델  : {RUN_MODELS}")
log("BATCH", f"예측일: {FORECAST_DATE} | min_obs: {RUN_MIN_OBS}")
log("BATCH", "=" * 70)

for i, ticker in enumerate(tickers, 1):
    pct = i / total * 100
    log("PROGRESS", f"[{i:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

    # ── STEP 1 : 데이터 추출 ──────────────────────────────
    try:
        _src_df = fetch_financial_series(
            engine, ticker, RUN_ITEM, RUN_MIN_OBS
        )
    except ValueError as e:
        err_msg = str(e)
        if "음수 매출" in err_msg:
            # 음수 매출 → 예측 제외 (별도 카운트)
            log(ticker, f"[NEG-SKIP] {err_msg}")
            neg_skipped += 1
            neg_skip_list.append(ticker)
        else:
            log(ticker, f"[SKIP] {err_msg}")
            skipped += 1
            skip_list.append(ticker)
        continue
    except Exception as e:
        log(ticker, f"[SKIP] {e}")
        skipped += 1
        skip_list.append(ticker)
        continue

    _y = _src_df.set_index("date")["value"].copy()
    _y.index = pd.DatetimeIndex(_y.index)
    _y.name  = RUN_ITEM
    log(ticker, f"  {len(_y)}분기 | {_y.index[0].date()} ~ {_y.index[-1].date()}")

    # ── STEP 2 : 예측 ─────────────────────────────────────
    try:
        _fc_results = forecast_one_ticker(_y, ticker, RUN_HORIZON, RUN_MODELS)
    except Exception as e:
        log(ticker, f"[ERROR] 예측: {e}")
        traceback.print_exc()
        errored += 1
        error_list.append(ticker)
        del _src_df, _y
        clear_memory()
        continue

    # ── STEP 3 : 예측 인덱스 생성 ────────────────────────
    _freq     = infer_freq_alias(_y.index)
    _fc_index = make_forecast_index(_y.index[-1], RUN_HORIZON, _freq)

    # ── STEP 4 : Long-format 변환 ─────────────────────────
    try:
        _ldf = build_long_df(
            ticker           = ticker,
            item             = RUN_ITEM,
            src_df           = _src_df,
            forecast_results = _fc_results,
            forecast_index   = _fc_index,
            forecast_date    = FORECAST_DATE,
        )
    except Exception as e:
        log(ticker, f"[ERROR] Long-format 변환: {e}")
        errored += 1
        error_list.append(ticker)
        del _src_df, _y, _fc_results
        clear_memory()
        continue

    # ── STEP 5 : DB 저장 (1개씩 즉시 저장 — 메모리 최소화) ─
    try:
        save_to_db(engine, _ldf)
        success += 1
    except Exception as e:
        log(ticker, f"[ERROR] DB 저장: {e}")
        errored += 1
        error_list.append(ticker)

    # ── STEP 6 : 메모리 해제 ─────────────────────────────
    del _src_df, _y, _fc_results, _ldf
    clear_memory()

# ── 요약 ──────────────────────────────────────────────────
log("BATCH", "=" * 70)
log("BATCH", f"완료 | 성공: {success}  데이터스킵: {skipped}  음수제외: {neg_skipped}  오류: {errored}  합계: {total}")
if skip_list:     log("BATCH", f"데이터스킵  : {skip_list}")
if neg_skip_list: log("BATCH", f"음수매출제외 : {neg_skip_list}")
if error_list:    log("BATCH", f"오류 티커   : {error_list}")
log("BATCH", "=" * 70)


[모드] 특정 티커 지정: ['STRL', 'SIMO', 'APH', 'ISSC', 'FIX', 'DY']
[OK] 테이블 'us_revenue_forecast_data' 준비 완료
[BATCH] ======================================================================
[BATCH] 시작  | 티커 6개 | 항목: sale | 예측기간: 8분기
[BATCH] 모델  : ['SARIMA', 'ETS', 'Theta']
[BATCH] 예측일: 2026-04-30 | min_obs: 28
[BATCH] ======================================================================
[PROGRESS] [   1/6] ( 16.7%)  >>  STRL
[STRL]   40분기 | 2016-03-31 ~ 2025-12-31
[STRL]   [SARIMA] 시작  (메모리: 1449.1 MB)
[메모리] forecast_sarima 실행 전: 1449.07 MB
[메모리] find_best_sarima_params 실행 전: 1449.07 MB
[메모리] find_best_sarima_params 실행 후: 1449.07 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 후: 1449.07 MB (변화: +0.00 MB)
[STRL]   [SARIMA] 완료  첫값=7.29e+08 (메모리: 1449.1 MB)
[STRL]   [ETS] 시작  (메모리: 1449.1 MB)
[메모리] forecast_ets 실행 전: 1449.07 MB
[메모리] forecast_ets 실행 후: 1449.07 MB (변화: +0.00 MB)
[STRL]   [ETS] 완료  첫값=5.80e+08 (메모리: 1449.1 MB)
[STRL]   [Theta] 시작  (메모리: 1449.1 MB)
[메모리] forecast_theta 실행 전: 1449.07 MB


## Cell 14 · 저장 결과 조회

배치 완료 후 DB 에 저장된 결과를 확인합니다.

In [30]:
# ── 오늘 예측된 티커 × 모델별 요약 ──────────────────────────
with engine.connect() as conn:
    summary_df = pd.read_sql(
        text(f"""
            SELECT
                ticker,
                item,
                model,
                MIN(date)  AS date_from,
                MAX(date)  AS date_to,
                COUNT(*)   AS row_count,
                forecast_date
            FROM   `{DEST_TABLE}`
            WHERE  forecast_date = :fd
            GROUP  BY ticker, item, model, forecast_date
            ORDER  BY ticker, model
        """),
        conn,
        params={"fd": FORECAST_DATE},
    )

print(f"[오늘({FORECAST_DATE}) 예측 저장 결과: {len(summary_df)}건]")
display(summary_df)


[오늘(2026-04-30) 예측 저장 결과: 2644건]


,ticker,item,model,date_from,date_to,row_count,forecast_date
0,A,sale,actual,2016-04-30,2026-01-31,40,2026-04-30
1,A,sale,Ensemble,2026-03-31,2027-12-31,8,2026-04-30
2,A,sale,ETS,2026-03-31,2027-12-31,8,2026-04-30
3,A,sale,SARIMA,2026-03-31,2027-12-31,8,2026-04-30
4,A,sale,Theta,2026-03-31,2027-12-31,8,2026-04-30
...,...,...,...,...,...,...,...
2639,ZTS,sale,actual,2016-04-03,2025-12-31,40,2026-04-30
2640,ZTS,sale,Ensemble,2026-03-31,2027-12-31,8,2026-04-30
2641,ZTS,sale,ETS,2026-03-31,2027-12-31,8,2026-04-30
2642,ZTS,sale,SARIMA,2026-03-31,2027-12-31,8,2026-04-30


In [31]:
# ── 전체 DB 저장 통계 ─────────────────────────────────────
with engine.connect() as conn:
    total_stat = pd.read_sql(
        text(f"""
            SELECT
                forecast_date,
                COUNT(DISTINCT ticker) AS ticker_cnt,
                COUNT(DISTINCT model)  AS model_cnt,
                COUNT(*)               AS total_rows
            FROM   `{DEST_TABLE}`
            GROUP  BY forecast_date
            ORDER  BY forecast_date DESC
            LIMIT  10
        """),
        conn,
    )

print("[전체 DB 저장 현황 (최근 10개 예측일)]")
display(total_stat)


[전체 DB 저장 현황 (최근 10개 예측일)]


,forecast_date,ticker_cnt,model_cnt,total_rows
0,2026-04-30,489,7,36792
1,2026-04-29,1,7,88
2,2026-04-27,1,7,88
3,2026-04-24,1,7,88
4,2026-04-04,444,7,38893
5,2026-04-03,1421,7,124293
6,2026-03-31,731,7,61033
7,2026-03-30,964,7,79894


## Cell 15 · 오염 데이터 삭제 & 재예측

### 왜 필요한가?
기존 예측은 **FMP 중복 데이터(같은 실적이 다른 분기 날짜에 저장된 행)**를  
그대로 포함한 시계열로 예측했습니다.  
예) `date_month=2025-12`인 실적이 `2025-12-31`과 `2026-03-31` 두 날짜에 저장  
→ 시계열 마지막에 **가짜 분기가 추가**되어 예측 기준점이 한 분기 뒤로 밀림

### 처리 순서
1. **Cell 15-A** : 삭제 대상 확인 (실제 삭제 전 확인용)
2. **Cell 15-B** : `us_revenue_forecast_data` 에서 기존 예측 데이터 삭제
3. **Cell 13** : 수정된 `fetch_financial_series` 로 재예측 실행

> ⚠️ 특정 ticker 만 재예측하려면 `RUN_TICKERS = ["AAPL", ...]` 로 지정하세요.

In [23]:
# # ══════════════════════════════════════════════════════
# #  Cell 15-A : 삭제 대상 확인 (읽기 전용 — 실제 삭제 안 함)
# # ══════════════════════════════════════════════════════
#
# # 삭제할 forecast_date 지정
# # None → DEST_TABLE 전체 삭제 / 문자열 → 특정 날짜만
# DELETE_FORECAST_DATE = None    # 예: "2026-03-25"  또는 None (전체)
# DELETE_TICKERS       = None    # 예: ["AAPL", "MSFT"]  또는 None (전체)
#
# # ── 삭제 대상 row 수 확인 ─────────────────────────────
# with engine.connect() as conn:
#     cond_parts = []
#     cond_params = {}
#     if DELETE_FORECAST_DATE:
#         cond_parts.append("forecast_date = :fd")
#         cond_params["fd"] = DELETE_FORECAST_DATE
#     if DELETE_TICKERS:
#         in_clause = ", ".join([f":t{i}" for i in range(len(DELETE_TICKERS))])
#         cond_parts.append(f"ticker IN ({in_clause})")
#         for i, t in enumerate(DELETE_TICKERS):
#             cond_params[f"t{i}"] = t
#
#     where_sql = ("WHERE " + " AND ".join(cond_parts)) if cond_parts else ""
#     check_sql = text(f"SELECT COUNT(*) AS cnt FROM `{DEST_TABLE}` {where_sql}")
#     row = conn.execute(check_sql, cond_params).fetchone()
#     cnt = row[0] if row else 0
#
# print(f"[확인] 삭제 대상 조건:")
# print(f"       forecast_date = {DELETE_FORECAST_DATE or '전체'}")
# print(f"       tickers       = {DELETE_TICKERS or '전체'}")
# print(f"       삭제 예정 행 수: {cnt:,}행")
# print()
# print("실제 삭제하려면 Cell 15-B 를 실행하세요.")


In [17]:
# # ══════════════════════════════════════════════════════
# #  Cell 15-B : 실제 삭제 실행
# #  ⚠️  되돌릴 수 없습니다. Cell 15-A 확인 후 실행하세요.
# # ══════════════════════════════════════════════════════
#
# # Cell 15-A 와 동일한 조건 사용
# with engine.begin() as conn:
#     del_parts = []
#     del_params = {}
#     if DELETE_FORECAST_DATE:
#         del_parts.append("forecast_date = :fd")
#         del_params["fd"] = DELETE_FORECAST_DATE
#     if DELETE_TICKERS:
#         in_clause = ", ".join([f":t{i}" for i in range(len(DELETE_TICKERS))])
#         del_parts.append(f"ticker IN ({in_clause})")
#         for i, t in enumerate(DELETE_TICKERS):
#             del_params[f"t{i}"] = t
#
#     where_sql = ("WHERE " + " AND ".join(del_parts)) if del_parts else ""
#     delete_sql = text(f"DELETE FROM `{DEST_TABLE}` {where_sql}")
#     result = conn.execute(delete_sql, del_params)
#
# print(f"[완료] 삭제된 행 수: {result.rowcount:,}행")
# print()
# print("다음 단계: Cell 13 을 실행해 재예측을 진행하세요.")
# print("  → fetch_financial_series 가 수정됐으므로 정확한 시계열로 재예측됩니다.")
